In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/monthly_sales.csv")
df["date"] = pd.to_datetime(df["date"])
ts = df.set_index("date")["sales"]

train_size = int(len(ts) * 0.8)
train = ts[:train_size]
test = ts[train_size:]

In [2]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    train,
    order=(1,1,1),
    seasonal_order=(1,1,1,12)
)

model_fit = model.fit()

C:\Users\SRIDHAR\Desktop\Demand_Forecasting_TimeSeries\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency M will be used.
  self._init_dates(dates, freq)
C:\Users\SRIDHAR\Desktop\Demand_Forecasting_TimeSeries\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency M will be used.
  self._init_dates(dates, freq)
C:\Users\SRIDHAR\Desktop\Demand_Forecasting_TimeSeries\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'


In [3]:
pred_sarima = model_fit.forecast(steps=len(test))

In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(test, pred_sarima)
rmse = np.sqrt(mean_squared_error(test, pred_sarima))
mape = np.mean(np.abs((test - pred_sarima) / test)) * 100

print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

MAE: 64401.15899211601
RMSE: 84183.06471219102
MAPE: 20.233685939415192


In [10]:
pred_df = pd.DataFrame({
    "date": test.index,
    "actual": test.values,
    "sarima_pred": pred_sarima.values
})

pred_df.to_csv("../outputs/metrics/sarima_predictions.csv", index=False)

In [5]:
!pip install joblib

Defaulting to user installation because normal site-packages is not writeable


In [8]:
import joblib
joblib.dump(model_fit, "../models/sarima.pkl")

['../models/sarima.pkl']